In [4]:
import pandas as pd
from data_profiling import ProfileReport

In [5]:
df = pd.read_csv("consolidation-etalab-schema-irve-statique-v-2.3.1-20260625.csv")

/tmp/ipykernel_52682/2572476926.py:1: DtypeWarning: Columns (12,18,19,20,21,22,24,25,29,33) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("consolidation-etalab-schema-irve-statique-v-2.3.1-20260625.csv")


In [6]:
df.columns

Index(['nom_amenageur', 'siren_amenageur', 'contact_amenageur',
       'nom_operateur', 'contact_operateur', 'telephone_operateur',
       'nom_enseigne', 'id_station_itinerance', 'id_station_local',
       'nom_station', 'implantation_station', 'adresse_station',
       'code_insee_commune', 'coordonneesXY', 'nbre_pdc', 'id_pdc_itinerance',
       'id_pdc_local', 'puissance_nominale', 'prise_type_ef', 'prise_type_2',
       'prise_type_combo_ccs', 'prise_type_chademo', 'prise_type_autre',
       'gratuit', 'paiement_acte', 'paiement_cb', 'paiement_autre',
       'tarification', 'condition_acces', 'reservation', 'horaires',
       'accessibilite_pmr', 'restriction_gabarit', 'station_deux_roues',
       'raccordement', 'num_pdl', 'date_mise_en_service', 'observations',
       'date_maj', 'cable_t2_attache', 'last_modified', 'datagouv_dataset_id',
       'datagouv_resource_id', 'datagouv_organization_or_owner', 'created_at',
       'consolidated_longitude', 'consolidated_latitude',
     

In [7]:
profile = ProfileReport(df, title="Profiling Report")

In [9]:
report_path = "rapport_data_profiling.html"
profile.to_file(report_path)

print(f"Rapport généré : {report_path}")

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 54.58it/s]

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 54.58it/s]

Rapport généré : rapport_data_profiling.html


In [11]:
def quality_score(df):
    total_cells = df.shape[0] * df.shape[1]

    missing = df.isna().sum().sum()
    completeness = 1 - missing / total_cells

    duplicates = df.duplicated().sum() / len(df)

    numeric_outliers = 0
    for col in df.select_dtypes("number"):
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        outliers = ((df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)).sum()
        numeric_outliers += outliers

    outlier_ratio = numeric_outliers / total_cells

    score = (
        0.5 * completeness +
        0.3 * (1 - duplicates) +
        0.2 * (1 - outlier_ratio)
    )

    return round(score * 100, 2)

print(quality_score(df))

93.77


93.77% des données semblent de qualité en se basant sur le nombre de doublons, de valeurs aberrantes et de valeurs manquantes.